# Week 9 · Topic 1 — The Attention Mechanism
### Demo Notebook: Scaled Dot-Product Attention, from Scratch (Pure Python + NumPy)

**Goal:** build the attention mechanism from Topic 1's slides yourself — no PyTorch, no
`torch.nn.MultiheadAttention`, just NumPy and the same four steps every time: **Score → Scale → Normalise → Blend**.

We'll work through three parts:
1. **A tiny, hand-traceable example** — the exact 2-number example from Slide 12, so you can check the code against numbers you can verify by hand.
2. **A toy sentence** — `"I am going"` — the same running example used on Slides 6, 8, and 9.
3. **The full running example** — `"I am going to the market"` (English) alongside its Yoruba translation `"Mo n lọ si ọja"` — this is the sentence Topics 2 and 4 will keep building on.

One concept per cell. Run them top to bottom.

In [ ]:
import numpy as np

# Keep printed numbers readable: 3 decimal places, no scientific notation
np.set_printoptions(precision=3, suppress=True)

print("NumPy ready. No other libraries needed for this notebook.")


NumPy ready. No other libraries needed for this notebook.


## Part 1 — The Tiny Hand-Traceable Example (Slide 12)

Before touching real sentences, let's reproduce the exact worked example from the slides:

- **Query** = (2, 0)
- **Key 1** = (2, 0), **Key 2** = (0, 2)
- **Value 1** = (1, 0), **Value 2** = (0, 1)

If our code is right, we should land on the same numbers as the slide: a score of **4 vs 0**,
scaled to roughly **2.83 vs 0**, normalised to roughly **95% vs 5%**, blending to an output very
close to **(0.95, 0.05)**.

In [ ]:
# Step 0 — set up the tiny example exactly as in Slide 12
Q = np.array([2.0, 0.0])          # one Query vector
K = np.array([[2.0, 0.0],         # two Key vectors (one per candidate word)
              [0.0, 2.0]])
V = np.array([[1.0, 0.0],         # two Value vectors (matching the two Keys)
              [0.0, 1.0]])

print("Query: ", Q)
print("Keys:\n", K)
print("Values:\n", V)


Query:  [2. 0.]
Keys:
 [[2. 0.]
 [0. 2.]]
Values:
 [[1. 0.]
 [0. 1.]]


In [ ]:
# Step 1 — Score: compare the Query against every Key with a dot product
scores = Q @ K.T
print("Raw scores:", scores)   # expect [4. 0.] to match the slide


Raw scores: [4. 0.]


In [ ]:
# Step 2 — Scale: divide by the square root of the key dimension
d_k = Q.shape[0]                      # dimension of the Query/Key vectors (2 here)
scaled_scores = scores / np.sqrt(d_k)
print(f"d_k = {d_k}, sqrt(d_k) = {np.sqrt(d_k):.3f}")
print("Scaled scores:", scaled_scores)   # expect roughly [2.828 0.]


d_k = 2, sqrt(d_k) = 1.414
Scaled scores: [2.828 0.   ]


In [ ]:
# Step 3 — Normalise: turn scaled scores into attention weights with softmax,
# written from scratch (no library softmax function)
def softmax(x):
    x = np.array(x, dtype=float)
    shifted = x - np.max(x)     # subtract the max first, purely for numerical stability
    exp = np.exp(shifted)
    return exp / np.sum(exp)

weights = softmax(scaled_scores)
print("Attention weights:", weights)          # expect roughly [0.944 0.056]
print("Weights sum to:", weights.sum())        # should always be 1.0


Attention weights: [0.944 0.056]
Weights sum to: 1.0


In [ ]:
# Step 4 — Blend: combine the Values using the attention weights
output = weights @ V
print("Blended output:", output)   # expect roughly [0.944 0.056], very close to the slide's (0.95, 0.05)


Blended output: [0.944 0.056]


**Check:** our code landed on `(0.944, 0.056)` — matching the slide's `(0.95, 0.05)`
(small rounding differences are expected since the slide uses 2 decimal places). The word whose
Key matched the Query most closely (score of 4 vs 0) dominates the blended output, exactly as
the slides predicted.

Now let's apply these same four steps to real words.

## Part 2 — Toy Sentence: `"I am going"` (Slides 6, 8, 9)

Real words don't come with Query/Key/Value vectors built in — we have to create them. This
section shows every step of that process:

1. Represent each token as a simple embedding vector
2. Project each embedding into a Query, a Key, and a Value vector
3. Run the same four steps as Part 1 — Score, Scale, Normalise, Blend — but now for a whole
   sentence at once, using matrices instead of single vectors

In [ ]:
# One concept per cell: represent tokens as simple numeric vectors.
# In a real model these come from a trained embedding layer — here we just
# hand-pick small numbers so every step stays easy to follow.
tokens_toy = ["I", "am", "going"]

embeddings_toy = {
    "I":     np.array([1.0, 0.0, 1.0, 0.0]),
    "am":    np.array([0.0, 1.0, 0.0, 1.0]),
    "going": np.array([1.0, 1.0, 0.0, 0.0]),
}

X_toy = np.stack([embeddings_toy[t] for t in tokens_toy])
print("Token order:", tokens_toy)
print("Embedding matrix (3 tokens x 4 dimensions):")
print(X_toy)


Token order: ['I', 'am', 'going']
Embedding matrix (3 tokens x 4 dimensions):
[[1. 0. 1. 0.]
 [0. 1. 0. 1.]
 [1. 1. 0. 0.]]


In [ ]:
# Define Query, Key, and Value projection matrices manually.
# Each embedding (length 4) will be projected into a Query/Key/Value vector (also length 4).
# In a real model these numbers are LEARNED during training — here we fix a random seed
# so the numbers are reproducible, and treat them as if they were already learned.
np.random.seed(42)
d_model = 4   # size of each token's embedding
d_k = 4       # size of each Query/Key/Value vector

W_Q = np.round(np.random.randn(d_model, d_k) * 0.5, 2)
W_K = np.round(np.random.randn(d_model, d_k) * 0.5, 2)
W_V = np.round(np.random.randn(d_model, d_k) * 0.5, 2)

print("W_Q (Query projection):\n", W_Q)
print("W_K (Key projection):\n", W_K)
print("W_V (Value projection):\n", W_V)


W_Q (Query projection):
 [[ 0.25 -0.07  0.32  0.76]
 [-0.12 -0.12  0.79  0.38]
 [-0.23  0.27 -0.23 -0.23]
 [ 0.12 -0.96 -0.86 -0.28]]
W_K (Key projection):
 [[-0.51  0.16 -0.45 -0.71]
 [ 0.73 -0.11  0.03 -0.71]
 [-0.27  0.06 -0.58  0.19]
 [-0.3  -0.15 -0.3   0.93]]
W_V (Value projection):
 [[-0.01 -0.53  0.41 -0.61]
 [ 0.1  -0.98 -0.66  0.1 ]
 [ 0.37  0.09 -0.06 -0.15]
 [-0.74 -0.36 -0.23  0.53]]


In [ ]:
# Project every token's embedding into its Query, Key, and Value vectors,
# all three tokens at once via matrix multiplication.
Q_toy = X_toy @ W_Q
K_toy = X_toy @ W_K
V_toy = X_toy @ W_V

print("Queries (one row per token):\n", Q_toy)
print("\nKeys (one row per token):\n", K_toy)
print("\nValues (one row per token):\n", V_toy)


Queries (one row per token):
 [[ 0.02  0.2   0.09  0.53]
 [ 0.   -1.08 -0.07  0.1 ]
 [ 0.13 -0.19  1.11  1.14]]

Keys (one row per token):
 [[-0.78  0.22 -1.03 -0.52]
 [ 0.43 -0.26 -0.27  0.22]
 [ 0.22  0.05 -0.42 -1.42]]

Values (one row per token):
 [[ 0.36 -0.44  0.35 -0.76]
 [-0.64 -1.34 -0.89  0.63]
 [ 0.09 -1.51 -0.25 -0.51]]


In [ ]:
# Step 1 — Score: compare every token's Query against every token's Key at once.
# Q_toy @ K_toy.T gives a 3x3 grid: row = the word asking, column = the word being compared to.
raw_scores = Q_toy @ K_toy.T
print("Raw attention scores (rows = Query word, columns = Key word):")
print("        I       am      going")
print(raw_scores)


Raw attention scores (rows = Query word, columns = Key word):
        I       am      going
[[-0.34   0.049 -0.776]
 [-0.218  0.322 -0.167]
 [-1.879  0.056 -2.066]]


In [ ]:
# Step 2 — Scale: divide every score by the square root of the Key dimension.
scaled_scores = raw_scores / np.sqrt(d_k)
print(f"Scaling factor: sqrt({d_k}) = {np.sqrt(d_k):.3f}")
print("Scaled scores:")
print(scaled_scores)


Scaling factor: sqrt(4) = 2.000
Scaled scores:
[[-0.17   0.024 -0.388]
 [-0.109  0.161 -0.083]
 [-0.94   0.028 -1.033]]


In [ ]:
# Step 3 — Normalise: apply softmax to EACH ROW so every word's attention weights sum to 1.
def softmax_rows(x):
    x = np.array(x, dtype=float)
    shifted = x - np.max(x, axis=-1, keepdims=True)   # stabilise each row independently
    exp = np.exp(shifted)
    return exp / np.sum(exp, axis=-1, keepdims=True)

weights_toy = softmax_rows(scaled_scores)
print("Attention weights (each row sums to 1):")
print("        I       am      going")
print(weights_toy)
print("\nRow sums (should all be 1.0):", weights_toy.sum(axis=1))


Attention weights (each row sums to 1):
        I       am      going
[[0.331 0.402 0.266]
 [0.3   0.393 0.308]
 [0.22  0.579 0.201]]

Row sums (should all be 1.0): [1. 1. 1.]


In [ ]:
# Step 4 — Blend: combine the Value vectors according to each word's attention weights.
output_toy = weights_toy @ V_toy
print("Context-aware output vectors (one row per token):")
for tok, vec in zip(tokens_toy, output_toy):
    print(f"  {tok:6s} -> {vec}")


Context-aware output vectors (one row per token):
  I      -> [-0.114 -1.087 -0.309 -0.134]
  am     -> [-0.116 -1.122 -0.321 -0.137]
  going  -> [-0.274 -1.176 -0.489  0.095]


**Reading the result:** each output row is no longer just that word's own embedding — it's
a blend of every word's Value, weighted by how strongly that word's Query matched each Key. This
mirrors Slide 9's point exactly: *"going"*'s new representation is a mix of its own content plus
context borrowed from *"I"* and *"am"* — just with numbers we generated ourselves rather than the
slide's illustrative table.

> **Note:** because we hand-picked different embeddings and weights than the slide deck's
> illustrative table, the exact percentages here won't match Slide 8's 9%/34%/57% — but the
> mechanism producing them is identical.

Now let's package this into one reusable function and run it on the full sentence — the version
Topics 2 and 4 will build on.

## Part 3 — The Full Running Example

**English:** `"I am going to the market"`
**Yoruba:** `"Mo n lọ si ọja"`

This is the sentence Topic 2 (multi-head attention + positional encoding) and Topic 4
(the full Transformer block) will keep reusing. To make that reuse easy, we'll first wrap
Parts 1–2's logic into a single function.

In [ ]:
# Package the four steps into one reusable function.
# Topic 2's notebook will import/reuse this exact function inside each attention head.
def scaled_dot_product_attention(X, W_Q, W_K, W_V):
    """
    Runs full self-attention on a sequence of token embeddings.

    X   : (num_tokens, d_model) matrix of token embeddings
    W_Q, W_K, W_V : (d_model, d_k) projection matrices

    Returns:
        output  : (num_tokens, d_k) context-aware output vectors
        weights : (num_tokens, num_tokens) attention weight matrix
    """
    Q = X @ W_Q
    K = X @ W_K
    V = X @ W_V

    d_k = Q.shape[-1]
    raw_scores = Q @ K.T
    scaled_scores = raw_scores / np.sqrt(d_k)
    weights = softmax_rows(scaled_scores)
    output = weights @ V
    return output, weights

print("scaled_dot_product_attention() is defined and ready to reuse.")


scaled_dot_product_attention() is defined and ready to reuse.


In [ ]:
# Tokenize the full running example. We keep the Yoruba translation alongside
# purely as reference context (per Topic 1, this notebook only computes SELF-attention
# within the English sentence — attention ACROSS the two languages is cross-attention,
# which is covered conceptually in Topic 3).
tokens_full = ["I", "am", "going", "to", "the", "market"]
yoruba_reference = "Mo n lọ si ọja"

print("English tokens:", tokens_full)
print("Yoruba translation (reference only):", yoruba_reference)


English tokens: ['I', 'am', 'going', 'to', 'the', 'market']
Yoruba translation (reference only): Mo n lọ si ọja


In [ ]:
# Give each of the 6 tokens a simple embedding vector (again, hand-picked/random
# here in place of a trained embedding layer).
np.random.seed(7)
embeddings_full = {t: np.round(np.random.randn(d_model), 2) for t in tokens_full}

print("Embeddings:")
for t in tokens_full:
    print(f"  {t:8s} -> {embeddings_full[t]}")

X_full = np.stack([embeddings_full[t] for t in tokens_full])
print("\nEmbedding matrix shape:", X_full.shape, "(6 tokens x 4 dimensions)")


Embeddings:
  I        -> [ 1.69 -0.47  0.03  0.41]
  am       -> [-0.79  0.   -0.   -1.75]
  going    -> [ 1.02  0.6  -0.63 -0.17]
  to       -> [ 0.51 -0.26 -0.24 -1.45]
  the      -> [ 0.55  0.12  0.27 -1.53]
  market   -> [ 1.65  0.15 -0.39  2.03]

Embedding matrix shape: (6, 4) (6 tokens x 4 dimensions)


In [ ]:
# Run the full self-attention pipeline on the sentence in one call,
# reusing the same W_Q, W_K, W_V projections from Part 2.
output_full, weights_full = scaled_dot_product_attention(X_full, W_Q, W_K, W_V)

print("Attention weight matrix (rows = Query word, columns = Key word):")
header = "         " + "".join(f"{t:>8s}" for t in tokens_full)
print(header)
for tok, row in zip(tokens_full, weights_full):
    print(f"{tok:8s} " + "".join(f"{v:8.3f}" for v in row))

print("\nEach row sums to:", np.round(weights_full.sum(axis=1), 3))


Attention weight matrix (rows = Query word, columns = Key word):
                I      am   going      to     the  market
I           0.158   0.185   0.139   0.116   0.108   0.294
am          0.126   0.227   0.147   0.231   0.193   0.076
going       0.114   0.305   0.134   0.145   0.108   0.195
to          0.121   0.289   0.136   0.188   0.143   0.124
the         0.122   0.297   0.130   0.192   0.138   0.122
market      0.157   0.130   0.140   0.083   0.089   0.401

Each row sums to: [1. 1. 1. 1. 1. 1.]


In [ ]:
# Read off, for each word, which other word it attends to most strongly.
print("Strongest attention target per word:")
for i, t in enumerate(tokens_full):
    j = np.argmax(weights_full[i])
    print(f"  {t:8s} -> attends most to '{tokens_full[j]}' ({weights_full[i][j]*100:.1f}%)")


Strongest attention target per word:
  I        -> attends most to 'market' (29.4%)
  am       -> attends most to 'to' (23.1%)
  going    -> attends most to 'am' (30.5%)
  to       -> attends most to 'am' (28.9%)
  the      -> attends most to 'am' (29.7%)
  market   -> attends most to 'market' (40.1%)


In [ ]:
# Finally, the context-aware output vectors — what actually gets passed forward
# to the rest of the model (multi-head attention and positional encoding, in Topic 2).
print("Final context-aware output vectors:")
for tok, vec in zip(tokens_full, output_full):
    print(f"  {tok:8s} -> {vec}")

print("\nOutput shape:", output_full.shape, "(same shape as the input embeddings — 6 tokens x 4 dimensions)")


Final context-aware output vectors:
  I        -> [-0.063 -0.505  0.348 -0.506]
  am       -> [ 0.575  0.007  0.418 -0.755]
  going    -> [ 0.3   -0.156  0.324 -0.563]
  to       -> [ 0.477 -0.027  0.368 -0.66 ]
  the      -> [ 0.488 -0.007  0.369 -0.659]
  market   -> [-0.366 -0.774  0.324 -0.408]

Output shape: (6, 4) (same shape as the input embeddings — 6 tokens x 4 dimensions)


## Wrap-Up

In this notebook we:

- Reproduced the exact tiny hand-traceable example from Slide 12, confirming our code matches the slide's numbers
- Built Query, Key, and Value projections from scratch and ran the full four-step attention process — Score, Scale, Normalise, Blend — on a toy sentence
- Packaged the logic into one reusable `scaled_dot_product_attention()` function
- Ran that function on the full running example, `"I am going to the market"` (with `"Mo n lọ si ọja"` as the Yoruba reference translation), and inspected the resulting attention weights and output vectors

**Up next (Topic 2):** we'll reuse `scaled_dot_product_attention()` inside multiple attention
heads running in parallel, and add positional encoding so the model also knows word order —
something this notebook's plain self-attention has no sense of yet.